# Expérimentation — détection de fraude

Le notebook reprend les fonctions utilisées par le backend. Les résultats sont calculés à l’exécution, sans valeurs préremplies. Par défaut, on travaille sur le jeu de démonstration ; passer à `kaggle` pour les transactions Sparkov.

In [ ]:
import sys
from pathlib import Path
root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root / "backend"))

from app.services.datasets import dataset, overview
from app.services.modeling import prepare, start_training, train_async, status, evaluation
from fastapi import BackgroundTasks
import pandas as pd

In [ ]:
dataset_id = "demo"  # Remplacer par "kaggle" pour Sparkov
df = dataset(dataset_id)
overview(df)

In [ ]:
prepared = prepare(dataset_id)
[(len(part), int(part.is_fraud.sum())) for part in prepared["raw"]], prepared["split_method"], prepared["columns"]

## Modèles et seuil métier

La régression logistique compare les pondérations sur validation. Les modèles non supervisés sont ajustés uniquement sur les transactions légitimes de train. Les seuils minimisent le coût sur validation, puis sont appliqués tels quels sur test.

In [ ]:
tasks = BackgroundTasks()
job = start_training(dataset_id, budget=60, review_cost=15., background_tasks=tasks)
train_async(job["job_id"], dataset_id, budget=60, review_cost=15.)
run = status(job["job_id"])
run["state"]

In [ ]:
pd.DataFrame([{"Modèle": x["label"], "PR-AUC validation": x["val_pr_auc"],
               "PR-AUC test": x["test"]["pr_auc"], "Coût test": x["test_cost"]} for x in run["results"]])

In [ ]:
chosen = run["selected_model"]
rep = evaluation(job["job_id"], chosen, None, 60, 15.)
rep["metrics"], rep["cost"]

## Explication d’une alerte

Cette cellule utilise SHAP sur un petit échantillon ; elle nécessite `shap` et peut être plus lente. Les contributions ne prouvent pas une causalité.

In [ ]:
from app.services.xai import explain
exp = explain(job["job_id"], chosen, observation=0, global_count=4, background_size=6)
pd.DataFrame(exp["reasons"])